# Hibiki-Sw Stage 3: Speech-to-Speech Translation

**GPU Budget: ~8 hours** (40k steps across 1-2 sessions, 2x T4)

Train the full multi-stream Hibiki model on paired en<->sw audio.
The model learns to translate speech while preserving speaker
characteristics via the Inner Monologue text stream.

**Prerequisites:**
- Stage 2 checkpoint
- S2ST manifest from Notebook 00

In [ ]:
!pip install -q transformers accelerate sentencepiece pyyaml tensorboard

In [ ]:
import os
import torch

print(f"GPUs: {torch.cuda.device_count()}")

REPO_DIR = "/kaggle/input/hibiki-sw-code/hibiki-sw"
DATA_DIR = "/kaggle/input/hibiki-sw-data"
OUTPUT_DIR = "/kaggle/working/stage3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STAGE2_CKPT = "/kaggle/input/hibiki-sw-stage2/checkpoint_final.pt"
MANIFEST = f"{DATA_DIR}/manifests/fleurs_en2sw_train.tsv"

RESUME_CKPT = None
# RESUME_CKPT = "/kaggle/input/hibiki-sw-stage3-partial/checkpoint_20000.pt"

In [ ]:
!cp -r {REPO_DIR}/* /kaggle/working/hibiki-sw/

In [ ]:
%%time
resume_flag = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""

# Optional: HF Hub backup
hf_flags = ""
# hf_flags = "--hf_repo YOUR_USER/hibiki-sw-stage3 --hf_token YOUR_TOKEN"

!cd /kaggle/working/hibiki-sw && torchrun \
    --nproc_per_node=2 \
    --master_port=29500 \
    training/train_s2st.py \
    --config configs/model_100m.yaml \
    --manifest {MANIFEST} \
    --stage2_ckpt {STAGE2_CKPT} \
    --output_dir {OUTPUT_DIR} \
    {resume_flag} {hf_flags}

In [ ]:
# Check progress
import glob
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_*.pt"))
for c in ckpts:
    ckpt = torch.load(c, map_location="cpu")
    print(f"  {os.path.basename(c)}: step {ckpt['step']}")

print("\nUpload latest checkpoint for persistence before session ends!")